# User-level DP, corrected — WiDS ICU mortality

The original user-level DP runs added DP noise to BatchNorm running statistics,
which drove variances negative and made every run return NaN. Here the encoder
uses LayerNorm (parameters only), so DP-FedAvg perturbs nothing but trainable
parameters. The sweep covers ε ∈ {0.5, 1, 3, 5, 10} plus a no-DP reference, for
5, 10 and 30 communication rounds, on five seeds, within-site, with the
attacks run at every point.

Attach `fedlgbm-src` and the **WiDS Datathon 2020** competition. Expect 2.5–3 hours.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')

hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, 'fedlgbm not found — attach the fedlgbm-src dataset'
sys.path.insert(0, str(hits[0].parent.parent))

train_files = sorted(INPUT.glob('**/training_v2.csv'))
assert train_files, 'attach the WiDS Datathon 2020 competition'
DATA = train_files[0].parent
print('source:', hits[0].parent.parent)
print('data:', train_files[0])

# CPU on purpose: the paper's runs were CPU, and results are not bit-identical
# across devices, so a GPU here would make these numbers incomparable with them.
DEVICE = 'cpu'
print('device =', DEVICE)


In [ ]:
from fedlgbm import ExperimentConfig, setup_logging
from fedlgbm.ablations import run_multi_seed, sweep_privacy_utility
import traceback

setup_logging()

# Fail in seconds, not hours, if Kaggle mounted a stale fedlgbm-src version:
# the old code would ignore save_predictions and skip per-feature errors.
import dataclasses, inspect
from fedlgbm import attacks as _attacks
from fedlgbm.config import MLPConfig as _M
assert 'layer_norm' in {f.name for f in dataclasses.fields(_M)}, 'stale fedlgbm-src'
assert 'feature_names' in inspect.signature(_attacks.evaluate_privacy).parameters, 'stale fedlgbm-src'
print('fedlgbm-src version check passed')

def build(split_mode: str, name: str) -> ExperimentConfig:
    # Identical to the paper's multi-seed configuration (kaggle_wids_multiseed).
    cfg = ExperimentConfig()
    cfg.name = name
    cfg.output_dir = OUT
    cfg.device = DEVICE
    cfg.skip_privacy_attacks = True
    cfg.save_predictions = False

    cfg.data.domain = 'wids'
    cfg.data.data_dir = DATA
    cfg.data.file_glob = 'training_v2.csv'
    cfg.data.max_rows = None
    cfg.data.max_per_class = None
    cfg.data.split_mode = split_mode

    cfg.partition.scheme = 'natural'
    cfg.partition.min_samples_per_client = 200

    cfg.fed.n_rounds = 30
    cfg.fed.local_epochs = 2
    cfg.mlp.hidden_dims = (256, 128)
    cfg.mlp.embedding_dim = 64

    cfg.lgbm.class_weight = None
    cfg.lgbm.num_leaves = 255
    cfg.lgbm.n_estimators = 1000
    return cfg

SEEDS = (42, 43, 44, 45, 46)
SKIP = {'lstm', 'isolation_forest', 'stacked_mlp_lgbm', 'centralized_mlp_lgbm'}
ONLY_FED = SKIP | {'centralized_lgbm', 'fedavg_mlp_only', 'random_forest', 'logistic_regression'}
WITH_CENTRAL = SKIP | {'fedavg_mlp_only', 'random_forest', 'logistic_regression'}
REFERENCE = 'fed_hybrid_mlp_lgbm'
DONE = {}

def part(label, fn):
    # One failed analysis must not cost the others.
    t0 = time.time()
    print('=' * 70)
    print(label, flush=True)
    print('=' * 70)
    try:
        DONE[label] = fn()
        print(f'{label}: done in {(time.time() - t0) / 60:.1f} min', flush=True)
    except Exception:
        traceback.print_exc()
        DONE[label] = 'FAILED'

In [ ]:
# User-level DP on the update channel only (embeddings released without noise),
# LayerNorm encoder, five seeds, for three schedule lengths.
def run_userdp(rounds):
    cfg = build('random', f'userdp_rounds{rounds}')
    cfg.mlp.layer_norm = True
    cfg.fed.n_rounds = rounds
    cfg.skip_privacy_attacks = False
    return sweep_privacy_utility(cfg, epsilons=(0.5, 1.0, 3.0, 5.0, 10.0),
                                 seeds=SEEDS, skip=ONLY_FED, channels='updates')

for rounds in (5, 10, 30):
    part(f'userdp_rounds{rounds}', lambda r=rounds: run_userdp(r))

In [ ]:
for k, v in DONE.items():
    print(k, 'FAILED' if v == 'FAILED' else 'ok')
shutil.make_archive('/kaggle/working/userdp_outputs', 'zip', OUT)
print('archived /kaggle/working/userdp_outputs.zip')